# Start PHASE 5B


In [2]:
import pandas as pd
import networkx as nx

# load PPI edge list
ppi_df = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\PPI_TCGA_BRCA_STRING_HQ.csv"
)

# quick check
ppi_df.head(), ppi_df.shape


(               protein1              protein2  neighborhood  fusion  \
 0  9606.ENSP00000000233  9606.ENSP00000262812             0       0   
 1  9606.ENSP00000000233  9606.ENSP00000158762             0       0   
 2  9606.ENSP00000000233  9606.ENSP00000480707             0       0   
 3  9606.ENSP00000000233  9606.ENSP00000263245             0       0   
 4  9606.ENSP00000000233  9606.ENSP00000484121             0       0   
 
    cooccurence  coexpression  experimental  database  textmining  \
 0            0           190           163       600         173   
 1            0             0           147         0         736   
 2            0            98           187       600         194   
 3            0            63           391       600         527   
 4            0             0           519         0         566   
 
    combined_score gene1    gene2  
 0             745  ARF5     COPE  
 1             765  ARF5    ACAP1  
 2             731  ARF5    COPZ2  
 3    

# Build & clean the PPI graph

In [5]:
# build undirected PPI graph
G = nx.from_pandas_edgelist(
    ppi_df,
    source="gene1",
    target="gene2"
)

# remove self-loops (safety)
G.remove_edges_from(nx.selfloop_edges(G))

# basic graph stats
G.number_of_nodes(), G.number_of_edges()


(10095, 97248)

In [7]:
# get largest connected component
largest_cc = max(nx.connected_components(G), key=len)

# create subgraph
G_cc = G.subgraph(largest_cc).copy()

# check size
G_cc.number_of_nodes(), G_cc.number_of_edges()


(9875, 97117)

# Load seed genes & build personalization vector

In [10]:
import numpy as np

# load HER2 seed genes
her2_seeds = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Seeds_HER2.csv"
)["gene"].str.upper()

# keep seeds present in graph
her2_seeds = set(her2_seeds).intersection(G_cc.nodes())

len(her2_seeds)


3780

In [12]:
# build personalization vector
personalization = {node: 0 for node in G_cc.nodes()}
for g in her2_seeds:
    personalization[g] = 1

# normalize (important)
norm = sum(personalization.values())
personalization = {k: v / norm for k, v in personalization.items()}

# run personalized PageRank
pr_her2 = nx.pagerank(
    G_cc,
    alpha=0.85,
    personalization=personalization,
    max_iter=100
)

# convert to DataFrame
pr_her2_df = (
    pd.DataFrame.from_dict(pr_her2, orient="index", columns=["pagerank"])
    .sort_values("pagerank", ascending=False)
)

pr_her2_df.head(), pr_her2_df.shape


(        pagerank
 SRC     0.001975
 TP53    0.001818
 EGFR    0.001521
 RPS27A  0.001392
 EP300   0.001352,
 (9875, 1))

In [18]:
pr_her2_df.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\HER2_PPI_Pagerank.csv"
)

print("✅ Saved: HER2_PPI_Pagerank.csv")


✅ Saved: HER2_PPI_Pagerank.csv


# Luminal A PageRank

In [21]:
luma_seeds = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Seeds_LumA.csv"
)["gene"]

len(luma_seeds)


1052

In [23]:
# build personalization vector
personalization = {node: 0 for node in G_cc.nodes()}
for g in luma_seeds:
    if g in personalization:
        personalization[g] = 1

# normalize
norm = sum(personalization.values())
personalization = {k: v / norm for k, v in personalization.items()}

# run personalized PageRank
pr_luma = nx.pagerank(
    G_cc,
    alpha=0.85,
    personalization=personalization,
    max_iter=100
)

# convert to DataFrame
pr_luma_df = (
    pd.DataFrame.from_dict(pr_luma, orient="index", columns=["pagerank"])
    .sort_values("pagerank", ascending=False)
)

pr_luma_df.head(), pr_luma_df.shape


(       pagerank
 TP53   0.002233
 SRC    0.002190
 EP300  0.001869
 EGFR   0.001740
 HRAS   0.001672,
 (9875, 1))

In [25]:
pr_luma_df.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumA_PPI_Pagerank.csv"
)

print("✅ Saved: LumA_PPI_Pagerank.csv")


✅ Saved: LumA_PPI_Pagerank.csv


# Luminal B

In [28]:
lumb_seeds = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Seeds_LumB.csv"
)["gene"]

len(lumb_seeds)


3901

In [30]:
# build personalization vector
personalization = {node: 0 for node in G_cc.nodes()}
for g in lumb_seeds:
    if g in personalization:
        personalization[g] = 1

# normalize
norm = sum(personalization.values())
personalization = {k: v / norm for k, v in personalization.items()}

# run personalized PageRank
pr_lumb = nx.pagerank(
    G_cc,
    alpha=0.85,
    personalization=personalization,
    max_iter=100
)

# convert to DataFrame
pr_lumb_df = (
    pd.DataFrame.from_dict(pr_lumb, orient="index", columns=["pagerank"])
    .sort_values("pagerank", ascending=False)
)

pr_lumb_df.head(), pr_lumb_df.shape


(        pagerank
 SRC     0.001973
 TP53    0.001848
 EGFR    0.001528
 EP300   0.001395
 RPS27A  0.001390,
 (9875, 1))

In [32]:
pr_lumb_df.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\LumB_PPI_Pagerank.csv"
)

print("✅ Saved: LumB_PPI_Pagerank.csv")


✅ Saved: LumB_PPI_Pagerank.csv


# TNBC (Basal)

In [35]:
basal_seeds = pd.read_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\Seeds_TNBC.csv"
)["gene"]

len(basal_seeds)


3993

In [37]:
# build personalization vector
personalization = {node: 0 for node in G_cc.nodes()}
for g in basal_seeds:
    if g in personalization:
        personalization[g] = 1

# normalize
norm = sum(personalization.values())
personalization = {k: v / norm for k, v in personalization.items()}

# run personalized PageRank
pr_basal = nx.pagerank(
    G_cc,
    alpha=0.85,
    personalization=personalization,
    max_iter=100
)

# convert to DataFrame
pr_basal_df = (
    pd.DataFrame.from_dict(pr_basal, orient="index", columns=["pagerank"])
    .sort_values("pagerank", ascending=False)
)

pr_basal_df.head(), pr_basal_df.shape


(        pagerank
 SRC     0.001931
 TP53    0.001838
 EGFR    0.001459
 RPS27A  0.001389
 EP300   0.001337,
 (9875, 1))

In [39]:
pr_basal_df.to_csv(
    r"C:\Users\deep8\breast_cancer_project_folder\TNBC_PPI_Pagerank.csv"
)

print("✅ Saved: TNBC_PPI_Pagerank.csv")


✅ Saved: TNBC_PPI_Pagerank.csv
